# Phase 4: Global-Local Knowledge Augmentation (Hybrid RAG)

## Objective
Integrate the **Bitext Customer Support LLM Chatbot Training Dataset** into our existing vector store. This creates a "Global-Local" knowledge layer:
- **Local Domain Knowledge (LDK)**: Corporate-specific data (`customer_support_tickets.csv`).
- **Global Technical Knowledge (GTK)**: General customer support and technical categories from Bitext.

## Research Justification (IEEE Paper)
- **Hybrid Scaling**: Combining vertical (corporate) and horizontal (generic) datasets to improve F1-scores on fuzzy queries.
- **Intent Robustness**: Leveraging the Bitext dataset's diverse intent labeling to enrich retrieval context.

In [9]:
# Dependencies
import os
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from tqdm import tqdm

print("Libraries loaded. Ready for Global-Local augmentation.")

c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded. Ready for Global-Local augmentation.


## 1. Global Dataset Loading & Profiling (Bitext)
Loading the Bitext dataset as specified.

In [10]:
path = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
try:
    print(f"Fetching dataset from: {path}")
    ds = load_dataset(path)
    
    # Convert to DataFrame (using 'train' split)
    df_global_raw = pd.DataFrame(ds['train'])
    
    print(f"Successfully loaded {len(df_global_raw)} rows.")
    
    print("\n--- Data Info ---")
    display(df_global_raw.info())
    
    print("\n--- Category Distribution ---")
    display(df_global_raw['category'].value_counts())
    
    print("\n--- Sample Rows ---")
    display(df_global_raw.head())

except Exception as e:
    print(f"Error loading dataset: {e}")

Fetching dataset from: bitext/Bitext-customer-support-llm-chatbot-training-dataset


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating train split: 100%|██████████| 26872/26872 [00:00<00:00, 122556.37 examples/s]


Successfully loaded 26872 rows.

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


None


--- Category Distribution ---


category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64


--- Sample Rows ---


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## 2. Ingestion & Standardization
Mapping Bitext schema (`instruction` -> query, `response` -> resolution) to match our corporate LDK format.

In [11]:
def standardize_bitext_data(df):
    if df.empty: return df
    
    # Standard RAG schema: query, resolution, category, source
    df_clean = df[['instruction', 'response', 'category']].copy()
    df_clean = df_clean.rename(columns={'instruction': 'query', 'response': 'resolution'})
    
    df_clean['source'] = 'OpenSource_GTK_Bitext'
    return df_clean

df_global = standardize_bitext_data(df_global_raw)
print(f"Standardized {len(df_global)} rows from Bitext knowledge base.")

Standardized 26872 rows from Bitext knowledge base.


## 3. Hybrid Indexing (LDK + GTK)
Merging Local Domain Knowledge (LDK) with Global Technical Knowledge (GTK) into a unified FAISS index.

In [12]:
try:
    # Load Local Domain Knowledge (LDK)
    df_corp = pd.read_csv('../data/raw/customer_support_tickets.csv')
    df_corp_clean = pd.DataFrame({
        'query': df_corp['Ticket Description'],
        'resolution': df_corp['Resolution'],
        'category': df_corp['Ticket Type'],
        'source': 'Corporate_LDK'
    })
    
    # Combine datasets
    combined_df = pd.concat([df_corp_clean, df_global], ignore_index=True).dropna(subset=['query'])
    print(f"Total Unified Knowledge Base: {len(combined_df)} records")
    
    # Vectorize
    print("Initializing embedding model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(combined_df['query'].tolist(), show_progress_bar=True)
    
    # Build FAISS Index
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings).astype('float32'))
    
    print("Global-Local Hybrid Index created successfully.")
except Exception as e:
    print(f"Error during indexing: {e}")

Total Unified Knowledge Base: 35341 records
Initializing embedding model...


Batches: 100%|██████████| 1105/1105 [02:03<00:00,  8.91it/s]


Global-Local Hybrid Index created successfully.


## 4. Test: Cross-Source Retrieval
Evaluating retrieval accuracy for generic requests handled by the global dataset.

In [13]:
test_queries = [
    "I have a question about my invoice and payment options.",
    "How can I track my order status?",
    "I want to cancel my subscription and get a refund."
]

for query in test_queries:
    query_vec = model.encode([query])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k=2)
    
    print(f"Query: {query}")
    for idx in indices[0]:
        res = combined_df.iloc[idx]
        print(f"  [Source: {res['source']}] | [Category: {res['category']}]")
        print(f"  Snippet: {str(res['query'])[:100]}...\n")
    print("-"*80)

Query: I have a question about my invoice and payment options.
  [Source: OpenSource_GTK_Bitext] | [Category: INVOICE]
  Snippet: getting invoice...

  [Source: OpenSource_GTK_Bitext] | [Category: INVOICE]
  Snippet: get invoice...

--------------------------------------------------------------------------------
Query: How can I track my order status?
  [Source: OpenSource_GTK_Bitext] | [Category: ORDER]
  Snippet: check order status...

  [Source: OpenSource_GTK_Bitext] | [Category: ORDER]
  Snippet: order status...

--------------------------------------------------------------------------------
Query: I want to cancel my subscription and get a refund.
  [Source: OpenSource_GTK_Bitext] | [Category: SUBSCRIPTION]
  Snippet: I want to cancel my subscription to the newsletter, help me...

  [Source: OpenSource_GTK_Bitext] | [Category: SUBSCRIPTION]
  Snippet: I need to cancel my damn newsletter subscription, help me...

-------------------------------------------------------------------